In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import StackingClassifier
from sklearn.naive_bayes import BernoulliNB
from sklearn.metrics import accuracy_score, f1_score

df = pd.read_csv("cleaned_dataset.csv")

X = df.drop(["target", "id", "painting", "feel_text", "food", "soundtrack"], axis=1)
y = df["target"]

numeric_cols = X.select_dtypes(include=np.number).columns.tolist()

X["feel_text"] = df["feel_text"].fillna("")
X["food"] = df["food"].fillna("")
X["soundtrack"] = df["soundtrack"].fillna("")

preprocess = ColumnTransformer(
    transformers=[
        ("num", Pipeline(steps=[("imputer", SimpleImputer(strategy="mean")), ("scaler", StandardScaler())]), numeric_cols),
        ("feel_txt", TfidfVectorizer(max_features=500, min_df=3), "feel_text"),
        ("food_txt", TfidfVectorizer(max_features=500, min_df=3), "food"),
        ("soundtrack_txt", TfidfVectorizer(max_features=500, min_df=3), "soundtrack"),
    ],
    remainder="drop"
)

text_preprocess = ColumnTransformer(
    transformers=[
        ("feel_bin", CountVectorizer(max_features=500, min_df=3, binary=True), "feel_text"),
        ("food_bin", CountVectorizer(max_features=500, min_df=3, binary=True), "food"),
        ("soundtrack_bin", CountVectorizer(max_features=500, min_df=3, binary=True), "soundtrack"),
    ],
    remainder="drop"
)

X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.25, random_state=42, stratify=y_train_val
)

print(f"Split sizes: train={len(X_train)}, val={len(X_val)}, test={len(X_test)}")

Split sizes: train=966, val=322, test=322


In [2]:
from sklearn.model_selection import RepeatedStratifiedKFold, cross_validate

stack_tuning_results = []

bnb_alpha_values = [0.001, 0.01, 0.1]
lr_base_c_values = [0.01, 0.05, 0.1]
final_c_values = [0.1, 0.5, 1.0]

selection_cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=2, random_state=42)
scoring = {"acc": "accuracy", "f1_macro": "f1_macro"}

def build_stack_model(bnb_alpha, lr_c, final_c):
    return StackingClassifier(
        estimators=[
            (
                "bern_text",
                Pipeline([
                    ("preprocess", text_preprocess),
                    ("model", BernoulliNB(alpha=bnb_alpha, binarize=0.0, fit_prior=True)),
                ]),
            ),
            (
                "log_reg_full",
                Pipeline([
                    ("preprocess", preprocess),
                    ("model", LogisticRegression(C=lr_c, penalty="l2", solver="lbfgs", max_iter=2000)),
                ]),
            ),
        ],
        final_estimator=LogisticRegression(C=final_c, penalty="l2", solver="lbfgs", max_iter=2000),
        stack_method="predict_proba",
        cv=5,
        n_jobs=-1,
    )

best_stack_cv = {
    "mean_acc": -np.inf,
    "mean_f1_macro": -np.inf,
    "std_acc": 0.0,
    "std_f1_macro": 0.0,
    "params": {},
}

total_iterations = len(bnb_alpha_values) * len(lr_base_c_values) * len(final_c_values)

for iteration_count, (bnb_alpha, lr_c, final_c) in enumerate(
    [(a, b, c) for a in bnb_alpha_values for b in lr_base_c_values for c in final_c_values],
    start=1,
):
    tuned_stack = build_stack_model(bnb_alpha, lr_c, final_c)
    cv_out = cross_validate(
        tuned_stack,
        X_train_val,
        y_train_val,
        cv=selection_cv,
        scoring=scoring,
        n_jobs=-1,
        return_train_score=False,
    )

    mean_acc = float(np.mean(cv_out["test_acc"]))
    mean_f1 = float(np.mean(cv_out["test_f1_macro"]))
    std_acc = float(np.std(cv_out["test_acc"], ddof=1))
    std_f1 = float(np.std(cv_out["test_f1_macro"], ddof=1))

    stack_tuning_results.append(
        {
            "bnb_alpha": bnb_alpha,
            "lr_c": lr_c,
            "final_c": final_c,
            "cv_mean_accuracy": mean_acc,
            "cv_mean_f1_macro": mean_f1,
            "cv_std_accuracy": std_acc,
            "cv_std_f1_macro": std_f1,
        }
    )

    if (
        (mean_f1 > best_stack_cv["mean_f1_macro"])
        or (
            np.isclose(mean_f1, best_stack_cv["mean_f1_macro"])
            and mean_acc > best_stack_cv["mean_acc"]
        )
    ):
        best_stack_cv = {
            "mean_acc": mean_acc,
            "mean_f1_macro": mean_f1,
            "std_acc": std_acc,
            "std_f1_macro": std_f1,
            "params": {
                "bnb_alpha": bnb_alpha,
                "lr_c": lr_c,
                "final_c": final_c,
            },
        }

    print(f"{iteration_count}/{total_iterations}")

print("\nBest Stacked:")
print(f"CV mean accuracy: {best_stack_cv['mean_acc']:.4f}")
print(f"CV mean macro-F1: {best_stack_cv['mean_f1_macro']:.4f}")
print(f"BernoulliNB alpha: {best_stack_cv['params']['bnb_alpha']}")
print(f"Base Logistic C: {best_stack_cv['params']['lr_c']}")
print(f"Final Logistic C: {best_stack_cv['params']['final_c']}")

stack_tuning_df = pd.DataFrame(stack_tuning_results).sort_values(
    by=["cv_mean_f1_macro", "cv_mean_accuracy"],
    ascending=False,
)
display(stack_tuning_df.head(10))

1/27
2/27
3/27
4/27
5/27
6/27
7/27
8/27
9/27
10/27
11/27
12/27
13/27
14/27
15/27
16/27
17/27
18/27
19/27
20/27
21/27
22/27
23/27
24/27
25/27
26/27
27/27

Best Stacked:
CV mean accuracy: 0.9057
CV mean macro-F1: 0.9055
BernoulliNB alpha: 0.1
Base Logistic C: 0.1
Final Logistic C: 0.1


,bnb_alpha,lr_c,final_c,cv_mean_accuracy,cv_mean_f1_macro,cv_std_accuracy,cv_std_f1_macro
24,0.100,0.10,0.1,0.905666,0.905492,0.015529,0.015505
6,0.001,0.10,0.1,0.905656,0.905332,0.019591,0.019836
7,0.001,0.10,0.5,0.904491,0.904232,0.019170,0.019424
22,0.100,0.05,0.5,0.904108,0.903932,0.015569,0.015564
15,0.010,0.10,0.1,0.902939,0.902651,0.018618,0.018716
3,0.001,0.05,0.1,0.902932,0.902497,0.022414,0.022788
25,0.100,0.10,0.5,0.902559,0.902405,0.014701,0.014724
23,0.100,0.05,1.0,0.902552,0.902337,0.015955,0.015994
8,0.001,0.10,1.0,0.902552,0.902333,0.019271,0.019490
4,0.001,0.05,0.5,0.902543,0.902222,0.020262,0.020515


In [3]:
selected_params = best_stack_cv["params"]

best_bnb_alpha = selected_params["bnb_alpha"]
best_lr_c = selected_params["lr_c"]
best_final_c = selected_params["final_c"]

final_stack = StackingClassifier(
    estimators=[
        (
            "bern_text",
            Pipeline(
                [
                    ("preprocess", text_preprocess),
                    ("model", BernoulliNB(alpha=best_bnb_alpha, binarize=0.0, fit_prior=True)),
                ]
            ),
        ),
        (
            "log_reg_full",
            Pipeline(
                [
                    ("preprocess", preprocess),
                    ("model", LogisticRegression(C=best_lr_c, penalty="l2", solver="lbfgs", max_iter=2000)),
                ]
            ),
        ),
    ],
    final_estimator=LogisticRegression(C=best_final_c, penalty="l2", solver="lbfgs", max_iter=2000),
    stack_method="predict_proba",
    cv=5,
    n_jobs=-1,
)

final_stack.fit(X_train_val, y_train_val)
test_preds = final_stack.predict(X_test)
test_acc = accuracy_score(y_test, test_preds)
test_f1 = f1_score(y_test, test_preds, average="macro")

print(f"CV mean accuracy (selection): {best_stack_cv['mean_acc']:.4f}")
print(f"CV mean macro-F1 (selection): {best_stack_cv['mean_f1_macro']:.4f}")
print(f"\nTest accuracy: {test_acc:.4f}")
print(f"Test macro-F1: {test_f1:.4f}")

CV mean accuracy (selection): 0.9057
CV mean macro-F1 (selection): 0.9055

Test accuracy: 0.9317
Test macro-F1: 0.9316


C:\Users\rayya.DESKTOP-IQDDGR4\AppData\Roaming\Python\Python313\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(


In [4]:
from sklearn.base import clone

X_dev = X_train_val.copy()
y_dev = y_train_val.copy()

cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=10, random_state=42)
scoring = {"acc": "accuracy", "f1_macro": "f1_macro"}

# Final model with tuned hyperparameters
stack_nb_lr = StackingClassifier(
    estimators=[
        ("bern_text", Pipeline([
            ("preprocess", text_preprocess),
            ("model", BernoulliNB()),
        ])),
        ("log_reg_full", Pipeline([
            ("preprocess", preprocess),
            ("model", LogisticRegression(max_iter=2000)),
        ])),
    ],
    final_estimator=LogisticRegression(max_iter=2000),
    stack_method="predict_proba",
    cv=5,
    n_jobs=-1,
 )

cv_out = cross_validate(
    clone(stack_nb_lr),
    X_dev,
    y_dev,
    cv=cv,
    scoring=scoring,
    n_jobs=-1,
    return_train_score=False,
 )

stack_estimate = {
    "model": "stacked_bernoulli_plus_logreg",
    "point_estimate_accuracy": float(np.mean(cv_out["test_acc"])),
    "mean_f1_macro": float(np.mean(cv_out["test_f1_macro"])),
    "std_accuracy": float(np.std(cv_out["test_acc"], ddof=1)),
    "std_f1_macro": float(np.std(cv_out["test_f1_macro"], ddof=1)),
    "num_folds_total": len(cv_out["test_acc"]),
}

estimate_df = pd.DataFrame([stack_estimate])
print("Point estimate for expected test performance")
display(estimate_df)

print(
    f"stacked_bernoulli_plus_logreg: expected test accuracy = "
    f"{stack_estimate['point_estimate_accuracy']:.4f} "
    f"(CV std={stack_estimate['std_accuracy']:.4f}, "
    f"mean macro-F1={stack_estimate['mean_f1_macro']:.4f}, "
    f"folds={int(stack_estimate['num_folds_total'])})"
 )

Point estimate for expected test performance


,model,point_estimate_accuracy,mean_f1_macro,std_accuracy,std_f1_macro,num_folds_total
0,stacked_bernoulli_plus_logreg,0.904505,0.904284,0.021179,0.021369,50


stacked_bernoulli_plus_logreg: expected test accuracy = 0.9045 (CV std=0.0212, mean macro-F1=0.9043, folds=50)


In [5]:
# Export params logreg param
import json
import numpy as np

# Refit a dedicated log_reg pipeline so exported parameters are self-contained
logreg_export_pipe = Pipeline([
    ("preprocess", preprocess),
    ("model", LogisticRegression(max_iter=2000))
])
logreg_export_pipe.fit(X_train_val, y_train_val)

pre_fitted = logreg_export_pipe.named_steps["preprocess"]
lr_fitted = logreg_export_pipe.named_steps["model"]

num_pipe = pre_fitted.named_transformers_["num"]
imputer = num_pipe.named_steps["imputer"]
scaler = num_pipe.named_steps["scaler"]

feel_vec = pre_fitted.named_transformers_["feel_txt"]
food_vec = pre_fitted.named_transformers_["food_txt"]
soundtrack_vec = pre_fitted.named_transformers_["soundtrack_txt"]

np.savez(
    "logreg_params.npz",
    coef=np.array(lr_fitted.coef_, dtype=np.float64),
    intercept=np.array(lr_fitted.intercept_, dtype=np.float64),
    class_order=np.array(lr_fitted.classes_),
    numeric_cols=np.array(numeric_cols, dtype=object),
    num_means=np.array(imputer.statistics_, dtype=np.float64),
    num_stds=np.array(scaler.scale_, dtype=np.float64),
    train_indices=np.array(X_train.index),
    val_indices=np.array(X_val.index),
    train_val_indices=np.array(X_train_val.index),
    test_indices=np.array(X_test.index),
)

tfidf_meta = {
    "feel_text": {
        "vocab": {k: int(v) for k, v in feel_vec.vocabulary_.items()},
        "idf": [float(x) for x in feel_vec.idf_],
    },
    "food": {
        "vocab": {k: int(v) for k, v in food_vec.vocabulary_.items()},
        "idf": [float(x) for x in food_vec.idf_],
    },
    "soundtrack": {
        "vocab": {k: int(v) for k, v in soundtrack_vec.vocabulary_.items()},
        "idf": [float(x) for x in soundtrack_vec.idf_],
    },
}

with open("logreg_tfidf.json", "w", encoding="utf-8") as f:
    json.dump(tfidf_meta, f)

In [6]:
# Export params for manual Bernoulli Naive Bayes
import json
import numpy as np

bernoulli_export_pipe = Pipeline([
    ("preprocess", text_preprocess),
    ("model", BernoulliNB()),
])
bernoulli_export_pipe.fit(X_train_val, y_train_val)

pre_fitted = bernoulli_export_pipe.named_steps["preprocess"]
nb_fitted = bernoulli_export_pipe.named_steps["model"]

feel_vec = pre_fitted.named_transformers_["feel_bin"]
food_vec = pre_fitted.named_transformers_["food_bin"]
soundtrack_vec = pre_fitted.named_transformers_["soundtrack_bin"]

feature_log_prob = np.asarray(nb_fitted.feature_log_prob_, dtype=np.float64)
neg_log_prob = np.log(np.clip(1.0 - np.exp(feature_log_prob), 1e-12, 1.0))

np.savez(
    "bernoulli_params.npz",
    feature_log_prob=feature_log_prob,
    neg_log_prob=neg_log_prob,
    class_log_prior=np.asarray(nb_fitted.class_log_prior_, dtype=np.float64),
    class_order=np.asarray(nb_fitted.classes_),
)

bernoulli_vocab = {
    "feel_text": {
        "vocab": {k: int(v) for k, v in feel_vec.vocabulary_.items()},
        "size": len(feel_vec.vocabulary_),
        "offset": 0,
    },
    "food": {
        "vocab": {k: int(v) for k, v in food_vec.vocabulary_.items()},
        "size": len(food_vec.vocabulary_),
        "offset": len(feel_vec.vocabulary_),
    },
    "soundtrack": {
        "vocab": {k: int(v) for k, v in soundtrack_vec.vocabulary_.items()},
        "size": len(soundtrack_vec.vocabulary_),
        "offset": len(feel_vec.vocabulary_) + len(food_vec.vocabulary_),
    },
}

with open("bernoulli_vocab.json", "w", encoding="utf-8") as f:
    json.dump(bernoulli_vocab, f)


In [7]:
# Export stacked meta-model params
STACK_META_PATH = "stack_meta_params.npz"

nb_train_proba = bernoulli_export_pipe.predict_proba(X_train_val)
lr_train_proba = logreg_export_pipe.predict_proba(X_train_val)
stack_train_X = np.hstack([nb_train_proba, lr_train_proba])

stack_meta_model = LogisticRegression(max_iter=2000)
stack_meta_model.fit(stack_train_X, y_train_val)

np.savez(
    STACK_META_PATH,
    coef=np.asarray(stack_meta_model.coef_, dtype=np.float64),
    intercept=np.asarray(stack_meta_model.intercept_, dtype=np.float64),
    class_order=np.asarray(stack_meta_model.classes_),
    bernoulli_class_order=np.asarray(bernoulli_export_pipe.classes_),
    logreg_class_order=np.asarray(logreg_export_pipe.classes_),
    base_model_names=np.array(["bernoulli_nb", "logreg"], dtype=object),
)